# T04 — Provjera osnovnog modela u Google Colabu

**Cilj:** učitati `Qwen/Qwen2.5-1.5B-Instruct` u 4-bitnom NF4 režimu na Colab GPU-u i generisati jedan odgovor.

**Obavezno prije pokretanja:** `Runtime → Change runtime type → GPU`.

Notebook treba sačuvati kao:

`notebooks/01_environment_and_model_check.ipynb`


## 1. Provjera GPU-a

Ova ćelija mora prikazati NVIDIA GPU i `CUDA dostupna: True`.


In [1]:
!nvidia-smi

import sys
import torch

print("\nPython:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA dostupna:", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "GPU runtime nije aktivan. Uključi Runtime → Change runtime type → GPU."
)

print("GPU:", torch.cuda.get_device_name(0))


Thu Jul 23 10:48:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Kloniranje privatnog GitHub repozitorijuma

U Colab panelu **Secrets** dodaj secret pod nazivom `GITHUB_TOKEN` i omogući pristup notebooku.

Ne upisuj stvarnu vrijednost tokena u ćeliju.


In [2]:
from google.colab import userdata
from pathlib import Path
import os
import shutil
import subprocess

REPO_URL = "https://github.com/httpsinisha/bih-tourist-chatbot.git"
BRANCH = "feature/T04-colab-model-check"
REPO_DIR = Path("/content/bih-tourist-chatbot")

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError(
        "GITHUB_TOKEN nije dostupan. Dodaj ga u Colab Secrets i omogući pristup."
    )

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

private_url = REPO_URL.replace(
    "https://",
    f"https://x-access-token:{github_token}@",
)

result = subprocess.run(
    ["git", "clone", "--branch", BRANCH, private_url, str(REPO_DIR)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

if result.returncode != 0:
    raise RuntimeError(
        "Kloniranje nije uspjelo. Provjeri naziv grane i dozvole GitHub tokena."
    )

os.chdir(REPO_DIR)

print("Repozitorijum je kloniran.")
print("Radni direktorijum:", os.getcwd())
print("Aktivna grana:")
subprocess.run(["git", "branch", "--show-current"], check=True)


Repozitorijum je kloniran.
Radni direktorijum: /content/bih-tourist-chatbot
Aktivna grana:


CompletedProcess(args=['git', 'branch', '--show-current'], returncode=0)

## 3. Instalacija biblioteka

Instaliraju se biblioteke tražene u T04.


In [3]:
%pip install -q -U transformers datasets accelerate peft trl bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.0/885.0 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 9.0 MB/s eta 0:00:00


## 4. Provjera verzija i importa


In [4]:
import importlib.metadata as metadata

packages = [
    "transformers",
    "datasets",
    "accelerate",
    "peft",
    "trl",
    "bitsandbytes",
]

for package in packages:
    print(f"{package}: {metadata.version(package)}")


transformers: 5.14.1
datasets: 5.0.0
accelerate: 1.14.0
peft: 0.19.1
trl: 1.9.0
bitsandbytes: 0.49.2


## 5. Konfiguracija modela i generisanja

Koristi se osnovni model propisan projektom, seed 42 i 4-bitna NF4 kvantizacija.


In [5]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed,
)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
SEED = 42

set_seed(SEED)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("Model:", MODEL_NAME)
print("Seed:", SEED)
print("Kvantizacija: 4-bit NF4")


Model: Qwen/Qwen2.5-1.5B-Instruct
Seed: 42
Kvantizacija: 4-bit NF4


## 6. Učitavanje tokenizatora i modela

Učitavanje može trajati nekoliko minuta. Task prolazi samo ako nema CUDA out-of-memory greške.


In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
)

model.eval()

print("Tokenizer:", tokenizer.name_or_path)
print("Model config:", model.config._name_or_path)
print("Model type:", model.config.model_type)
print("4-bitno učitavanje:", getattr(model, "is_loaded_in_4bit", False))

device_map = getattr(model, "hf_device_map", None)

if device_map is not None:
    print("Raspored uređaja:", device_map)
else:
    print("Uređaj modela:", next(model.parameters()).device)

assert getattr(model, "is_loaded_in_4bit", False), (
    "Model nije učitan u 4-bitnom režimu."
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Tokenizer: Qwen/Qwen2.5-1.5B-Instruct
Model config: Qwen/Qwen2.5-1.5B-Instruct
Model type: qwen2
4-bitno učitavanje: True
Uređaj modela: cuda:0


## 7. Obavezni T04 upit

Ovo je pitanje navedeno u praktičnom planu. Slab ili netačan odgovor se ne ispravlja ručno — on predstavlja početno ponašanje osnovnog modela.


In [7]:
QUESTION = "Koja mjesta treba posjetiti u Bosni i Hercegovini?"

messages = [
    {
        "role": "system",
        "content": (
            "Ti si koristan turistički asistent. "
            "Odgovori na jeziku korisničkog pitanja."
        ),
    },
    {
        "role": "user",
        "content": QUESTION,
    },
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

model_inputs = tokenizer(
    [prompt],
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=300,
        temperature=0.2,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

new_token_ids = generated_ids[
    :,
    model_inputs["input_ids"].shape[1]:,
]

response = tokenizer.batch_decode(
    new_token_ids,
    skip_special_tokens=True,
)[0].strip()

print("Pitanje:")
print(QUESTION)
print("\nOdgovor:")
print(response)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Pitanje:
Koja mjesta treba posjetiti u Bosni i Hercegovini?

Odgovor:
Bosnia i Hercegovina je jedinstvena međunarodna zajednica, tako da nije potrebno posjetiti nekoliko mjesta za toj države. Sadašnji najpopulari putovanici su:

1. Sarajevo: Ovaj grad je centar Bosne i Hercegovine i ima mnogo interesnih otkrataćih muzeja.

2. Mostar: To je starih 500-letnijeg most koji se nalazi u Mostarskoj gradu.

3. Tuzla: Ovaj grad je odličan za njegovu prirodu i kulturnu povijestnu vježbu.

4. Zenica: Ovaj grad je popularan za njegovu prirodnu povijestnu vježbu.

5. Bratunak: Ovaj grad je odličan za njegovu prirodonu i kulturnu povijestnu vježbu.

6. Bijeljica: Ovaj grad je odličan za njegovu prirodonu i kulturnu povijestnu vježbu.

7. Travnik: Ovaj grad je odličan za njegovu prirodonu i kulturnu povijestnu vježbu.

8. Ban


## 8. Tehnička validacija

Ova provjera ne ocjenjuje kvalitet srpskog jezika. Provjerava samo da GPU radi, da je model učitan u 4-bitnom režimu i da odgovor nije prazan.


In [8]:
assert torch.cuda.is_available(), "CUDA nije dostupna."
assert getattr(model, "is_loaded_in_4bit", False), "Model nije 4-bitni."
assert isinstance(response, str) and response.strip(), "Odgovor je prazan."

print("T04 TEHNIČKA PROVJERA: USPJEŠNA")
print("✓ CUDA je dostupna")
print("✓ Model je učitan bez OOM greške")
print("✓ Model je učitan u 4-bitnom režimu")
print("✓ Generisan je neprazan odgovor")


T04 TEHNIČKA PROVJERA: USPJEŠNA
✓ CUDA je dostupna
✓ Model je učitan bez OOM greške
✓ Model je učitan u 4-bitnom režimu
✓ Generisan je neprazan odgovor


## 9. Zapažanje o osnovnom modelu

Model se uspješno učitava na Colab GPU-u i generiše odgovor, čime je tehnička provjera uspješna.

Osnovni model pokazuje slab kvalitet srpskog/BHS jezika i može miješati turističke lokacije. Ovaj rezultat se ne ispravlja ručno jer predstavlja početno stanje prije domain-specific fine-tuninga i RAG sistema.


## 10. Završna kontrolna lista

- [ ] `torch.cuda.is_available()` vraća `True`.
- [ ] U izlazu je sačuvan naziv GPU-a.
- [ ] Model se učitao bez CUDA out-of-memory greške.
- [ ] `4-bitno učitavanje` prikazuje `True`.
- [ ] Notebook prikazuje generisan odgovor na obavezno pitanje.
- [ ] U notebooku nema Hugging Face/GitHub tokena ni lozinki.
- [ ] Notebook je sačuvan sa svim izlazima kao `notebooks/01_environment_and_model_check.ipynb`.
